# 06 · Reprojection-Based Cloud Filtering  ← **Key visualisation**

**Goal:** clean the COLMAP sparse cloud by reprojecting every 3D point
into every frame and keeping only those that consistently land on
bark-coloured pixels. This is the proposal's core novelty — implemented
in `src/filter_cloud.py::filter_cloud_by_masks`.

The filter is the post-hoc cleanup that the LAB bark mask was attempting
(less effectively) at the COLMAP-pre-mask stage. Per the Week-3 ablation
in notebook 04, pre-masking COLMAP dropped frame registration from 79/80
to 43/80; the masks are far more useful applied AFTER reconstruction,
on 3D points whose multi-view-consistent reprojections are more
informative than any single-frame colour judgment.


In [ ]:
# Standard preamble.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline


## 0. Configuration

In [ ]:
from src import sfm, filter_cloud, segmentation, viz
import cv2

TREE_ID = "tree_5"

CLAHE_FRAMES_DIR = PROJECT_ROOT / "data" / "frames" / f"{TREE_ID}_clahe"
PLY              = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_sparse.ply"
POSES            = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_poses.npz"

for p, label in [(CLAHE_FRAMES_DIR, "CLAHE frames"), (PLY, "sparse PLY"), (POSES, "poses npz")]:
    if not p.exists():
        raise FileNotFoundError(f"{label} not found at {p}. Run notebook 04 first.")

print(f"Tree: {TREE_ID}")
print(f"Frames dir: {CLAHE_FRAMES_DIR}")
print(f"Cloud:      {PLY}")
print(f"Poses:      {POSES}")


## 1. Load the raw cloud, per-frame projection matrices, and generate per-frame masks

We re-generate the LAB bark masks here rather than reading saved binary
PNGs, so the filter can take advantage of the float mask (soft confidence)
without losing information to a 0/255 binary threshold.


In [ ]:
# Cloud + poses.
points_3d, _ = sfm.load_ply(PLY)
poses_data = np.load(POSES, allow_pickle=True)
Ps = list(poses_data["projections"])
image_names = list(poses_data["image_names"])
print(f"Cloud: {len(points_3d)} points")
print(f"Poses: {len(Ps)} cameras, image_names[0]={image_names[0]}")

# Per-frame bark masks — boolean (True = structure).
masks = []
for name in image_names:
    img_path = CLAHE_FRAMES_DIR / name
    if not img_path.exists():
        raise FileNotFoundError(f"Missing CLAHE frame: {img_path}")
    img = cv2.imread(str(img_path))
    mask_float = segmentation.bark_color_mask_lab(img)
    masks.append(mask_float > 0.5)
print(f"Generated {len(masks)} per-frame masks; first mask shape: {masks[0].shape}")


## 2. Apply the reprojection filter

For every 3D point: reproject into every frame; tally how many frames it
projects in-bounds AND lands on a `mask == True` pixel. Keep points whose
hit-rate ≥ `hit_rate_threshold` AND that are visible in ≥ `min_visible_frames`.


In [ ]:
kept_pts, kept_mask, diag = filter_cloud.filter_cloud_by_masks(
    points_3d, Ps, masks,
    hit_rate_threshold=0.6,
    min_visible_frames=5,
)
print(f"Survivors: {len(kept_pts)} / {len(points_3d)}  ({100*len(kept_pts)/len(points_3d):.1f}%)")
print(f"Visibility — median frames per point: {int(np.median(diag.visible_count))}, "
      f"max: {int(diag.visible_count.max())}")
print(f"Hit-rate of survivors — median: {float(np.median(diag.hit_rate[kept_mask])):.3f}, "
      f"min: {float(diag.hit_rate[kept_mask].min()):.3f}")


## 3. Optional second pass — statistical outlier removal

Drops points whose mean distance to their k nearest neighbours is anomalously
large (more than 2 standard deviations above the mean). Useful as a final
denoise after the reprojection filter has done the structural work.


In [ ]:
kept_pts_clean, outlier_mask = filter_cloud.remove_statistical_outliers(
    kept_pts, k=16, std_ratio=2.0,
)
print(f"After outlier removal: {len(kept_pts_clean)} / {len(kept_pts)} points kept")


## 4. Raw vs. filtered — the headline figure

In [ ]:
fig = viz.plot_cloud_pair(
    points_3d, kept_pts_clean,
    title_raw=f"Raw COLMAP cloud (N={len(points_3d)})",
    title_filtered=f"After reprojection filter + outlier removal (N={len(kept_pts_clean)})",
    suptitle=f"{TREE_ID} — reprojection filter "
             f"(hit-rate ≥ 0.6, visible in ≥ 5 frames; kept {100*len(kept_pts_clean)/len(points_3d):.1f}%)",
)
viz.save_fig(fig, f"06_{TREE_ID}_filter_comparison.png")
plt.show()


## 5. Save the filtered cloud for the skeleton stage

In [ ]:
out_ply = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_filtered.ply"
sfm.save_ply(out_ply, kept_pts_clean)
print(f"Filtered cloud → {out_ply}")

# Also save the diagnostics for the report's ablation discussion.
diag_npz = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_filter_diag.npz"
np.savez(
    diag_npz,
    visible_count=diag.visible_count,
    hit_count=diag.hit_count,
    hit_rate=diag.hit_rate,
    kept_mask=kept_mask,
)
print(f"Diagnostics → {diag_npz}")
